In [ ]:
from opendataval.dataloader import mix_labels_train
from opendataval.experiment import ExperimentMediator
import torch
from CA_RL_CODE import CARL

In [ ]:
import time

num_ber = 2

base_classes = [CARL]

class_suffixes = ['CARL_']

class_dict = {}
for suffix in class_suffixes:
    for i in range(1, num_ber):
        class_name = f"{suffix}{i}"
        base_class = base_classes[class_suffixes.index(suffix)]

        class_definition = {
            "__init__": lambda self, *args, **kwargs: super(self.__class__, self).__init__(*args, **kwargs)
        }

        new_class = type(class_name, (base_class,), class_definition)
        class_dict[class_name] = new_class


classes = [
    ('CARL', {}),
]
data_evaluators = []
for class_name, params in classes:
    for i in range(1, num_ber):
        random_seed = int(time.time())
        time.sleep(1)
        class_instance = class_dict[f'{class_name}_{i}'](**params, random_state=42)

        if class_name == "ds" or class_name == "bs":
            class_instance = class_dict[f'{class_name}_{i}'](**params, cache_name=f"{class_name}_{i}",
                                                             random_state=random_seed)

        data_evaluators.append(class_instance)

In [ ]:
dataset_name = "pol" #jannis:openML ID 43977;CPU:openML ID 761
train_count, valid_count, test_count = 1000, 300, 3000
noise_rate = 0.2
noise_kwargs = {'noise_rate': noise_rate}
train_kwargs = {}
model_name = "sklogreg"
metric_name = "accuracy"
output = 1

exper_med = ExperimentMediator.model_factory_setup(
    dataset_name=dataset_name,
    cache_dir=f"./output_{output}",
    force_download=False,
    train_count=train_count,
    valid_count=valid_count,
    test_count=test_count,
    add_noise=mix_labels_train,
    noise_kwargs=noise_kwargs,
    train_kwargs=train_kwargs,
    model_name=model_name,
    metric_name=metric_name,
    device=torch.device("cuda")
)

In [ ]:
# %%time
# compute data values.
exper_med = exper_med.compute_data_values(data_evaluators=data_evaluators)

In [ ]:
from opendataval.experiment.exper_methods import (
    discover_corrupted_sample,
    noisy_detection,
    remove_high_low,
    save_dataval, 
)
from matplotlib import pyplot as plt

# Saving the results
output_dir = f"./output_{output}/tmp/{dataset_name}_{noise_rate=}/"
exper_med.set_output_directory(output_dir)
output_dir

In [ ]:
exper_med.evaluate(save_dataval, save_output=True)
exper_med.evaluate(noisy_detection, save_output=True)


In [ ]:
fig = plt.figure(figsize=(15, 40))
_, fig = exper_med.plot(discover_corrupted_sample, fig, col=2, save_output=True)

In [ ]:
fig = plt.figure(figsize=(15, 30))
df_resp, fig = exper_med.plot(remove_high_low, fig, include_train=True, col=2, save_output=True)